# Lab 14: The Slow Ghost

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 2, *The Time Editor*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-14.ipynb)

**What you will do:** compare how fast a dim, low-contrast light or object seems to move against an identical bright one, log your own contrast-vs-lag judgements from the flashlight, Pulfrich-swing and on-screen versions, and check whether they follow the direction the lab predicts.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 14 you moved a flashlight, then dimmed it with cellophane and moved it at the same
speed: you may have noticed the dimmer light seeming to move more slowly. With a swinging
weight and a dark filter over one eye, a related but distinct effect turned flat side-to-side
motion into an illusory ellipse in depth: the classic **Pulfrich effect**, best explained by a
processing-speed difference between the two eyes that the brain's stereo system reads as a
depth offset. The flashlight's apparent slowing is usually explained differently, as a
Bayesian effect: noisier evidence from a dim or low-contrast object leans perception more
heavily on a built-in expectation that things are more often slow than fast.

## Record your results

The lag itself is only 10-20 milliseconds at the neural level, far too short to time with a
stopwatch. What you can log reliably is the strength of what you perceive: for each trial,
note how many layers of cellophane (or which sunglasses filter) you used, and rate how
strong the illusion felt, from 0 (no lag, no depth) to 5 (very clear lag, or a wide
ellipse).

In [ ]:
# example data: replace with your own ratings
trials = pd.DataFrame({
    "filter_layers": [0, 1, 2, 3, 0, 1, 2, 3],
    "trial_type":    ["flashlight", "flashlight", "flashlight", "flashlight",
                       "swinging weight", "swinging weight", "swinging weight", "swinging weight"],
    "rating": [0, 2, 3, 4, 0, 2, 3, 5],
})
display(trials)
trials.pivot_table(index="filter_layers", columns="trial_type", values="rating").plot.bar(figsize=(6, 3))
plt.xlabel("filter layers (more layers = dimmer)"); plt.ylabel("illusion strength (0-5)")
plt.xticks(rotation=0); plt.show()

## Compare

Lab 14 does not give a published rating scale or a number of layers to match; the claim it
makes is about direction: more dimming means noisier sensory evidence, and noisier evidence
means the brain's low-speed prior (for the flashlight) or the interocular latency difference
(for the swinging weight) has a stronger effect. Check whether your own ratings increase with
the number of filter layers, the way the lab's account predicts.

In [ ]:
trend = trials.groupby("filter_layers")["rating"].mean()
print(trend)
increasing = all(trend.iloc[i] <= trend.iloc[i + 1] for i in range(len(trend) - 1))
if increasing:
    print("Your ratings rise with dimming, consistent with the lab's account: more filtering means"
          " weaker, noisier evidence, which means a stronger illusion (via the low-speed prior for"
          " the flashlight, or a larger interocular latency difference for the swinging weight).")
else:
    print("Your ratings do not rise cleanly with dimming. Subjective 0-5 ratings are noisy, and"
          " cellophane layers do not dim light in precisely equal steps, so this is a rough test of"
          " direction, not a precise measurement -- rate a few more trials per layer count for a"
          " cleaner trend.")

## Pool the class data

Pool the class's ratings of the swinging-weight version specifically (Step 4), since it
gives the clearest yes/no percept: did you see an ellipse, or a flat line? Each row is one
anonymous ID and whether they saw the depth illusion with one layer of filter.

In [ ]:
import io
csv_text = """id,saw_ellipse
P01,1
P02,1
P03,0
P04,1
P05,1
P06,0
P07,1
P08,1
"""  # example data: replace with your class CSV, e.g. pd.read_csv("pulfrich_class.csv")
class_df = pd.read_csv(io.StringIO(csv_text))
p_hat = class_df["saw_ellipse"].mean()
boot = [class_df["saw_ellipse"].sample(len(class_df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Proportion who saw the depth illusion: {p_hat:.0%} "
      f"(95% bootstrap interval {lo:.0%}-{hi:.0%}, n = {len(class_df)})")
class_df["saw_ellipse"].value_counts().sort_index().plot.bar(color=["grey", "steelblue"], figsize=(4, 3))
plt.xticks([0, 1], ["flat line", "saw ellipse"], rotation=0); plt.ylabel("number of people"); plt.show()

## Questions to think about

1. The lab explains the flashlight illusion through a Bayesian low-speed prior, not through the motion-extrapolation account used for the flash-lag effect in Lab 12. Why would noisier evidence (from more layers of cellophane) be predicted to pull perceived speed further towards "slow," rather than simply making the object harder to see at all?
2. The lab explains the swinging-weight (Pulfrich) illusion through a difference in processing latency between your two eyes, which the stereo system reads as a difference in position rather than a difference in speed. Why does a position difference between the eyes get interpreted as depth, rather than as one eye simply "running behind" the other?
3. Carl Pulfrich, who first described the depth effect that bears his name, was blind in one eye and so could never see the illusion himself, relying instead on observations by his colleague Fertsch. What does this tell you about the relationship between understanding a mechanism and experiencing its effect?
4. The text lists sensor fusion (combining camera, lidar and radar) as the autonomous-vehicle answer to unreliable single-sensor signals, noting that lidar and radar, unlike cameras, are largely unaffected by darkness. What would the biological equivalent of "adding another sensor" be for a person struggling to judge the depth of a dim, fast-moving object?

## Challenge

Try the "Variation" from Step 4: type the same word in light grey and in black text in a word
processor, and scroll them at the same speed. Log your own rating of how much the grey word
appears to lag the black one, and compare it with your flashlight and swinging-weight ratings
above: do the three different versions of the illusion (light, motion-in-depth, on-screen
text) feel similarly strong to you, or does one stand out?

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-14.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")